In [5]:
import numpy as np
import os
import json
import duckdb
from fastapi import FastAPI
from pathlib import Path
from typing import List, Dict
import hashlib
import pip

In [6]:
print(f"Versão da biblioteca DuckDB: {duckdb.__version__}")

Versão da biblioteca DuckDB: 1.2.2


In [7]:
app = FastAPI()

# Configurações
EMBEDDINGS_FILE = "data/embeddings.duckdb"
JSON_SOURCE = "comments_1221_mistral.json"

def init_db() -> duckdb:
    """Inicializa o banco de dados DuckDB"""
    conn = duckdb.connect(EMBEDDINGS_FILE)
    conn.execute("""
    CREATE TABLE IF NOT EXISTS embeddings (
        id VARCHAR PRIMARY KEY,
        embedding FLOAT[],
        text TEXT,
        created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP
    )
    """)
    return conn

def load_embeddings_from_json(conn: duckdb):
    def generate_id(text: str) -> str:
        """Gera um ID único para cada comentário"""
        return hashlib.md5(text.encode()).hexdigest()

    """Carrega embeddings do JSON para o DuckDB"""
    if not Path(JSON_SOURCE).exists():
        return False
        
    with open(JSON_SOURCE) as f:
        data: Dict = json.load(f)
    
    embeddings = data.get("embedding", [])
    
    # Se você tiver os textos originais em outra lista
    texts = []  # Substitua por sua lista de textos se disponível
    with open(file=r'C:\Users\fuedj\Documents\Code\Embeddings_Cluster\comentarios.txt', mode='r', encoding='utf-8') as file:
        for linha in file:
            texts.append(linha)
    
    for idx, emb in enumerate(embeddings):
        text = texts[idx] if idx < len(texts) else f"embedding_{idx}"
        conn.execute("""
        INSERT OR IGNORE INTO embeddings (id, embedding, text)
        VALUES (?, ?, ?)
        """, [generate_id(text), emb, text])
    
    return True

In [8]:
conn = init_db()
load_embeddings_from_json(conn=conn)


True

In [9]:
# Verificação básica da estrutura
conn = duckdb.connect(EMBEDDINGS_FILE)
print(conn.execute("PRAGMA table_info('embeddings')").fetchdf())

   cid        name       type  notnull         dflt_value     pk
0    0          id    VARCHAR     True               None   True
1    1   embedding    FLOAT[]    False               None  False
2    2        text    VARCHAR    False               None  False
3    3  created_at  TIMESTAMP    False  CURRENT_TIMESTAMP  False


In [10]:
total_embeddings = conn.execute("SELECT COUNT(*) FROM embeddings").fetchone()[0]
print(f"Total de embeddings armazenados: {total_embeddings}")

Total de embeddings armazenados: 1209


In [11]:
sample = conn.execute("""
SELECT id, text, array_length(embedding) as dims 
FROM embeddings 
LIMIT 5
""").fetchdf()
print("\nAmostra de dados:")
print(sample)


Amostra de dados:
                                 id  \
0  cb0f2e74ebcbfb0a21f320fa6ef0b1eb   
1  a1b8f2efb813c2025ce7289e412e1ad2   
2  1e642a39f0b419accc96ed1fc67e416b   
3  a7e967f87c4765a321527844d01727c2   
4  b3967ad0b125798f6a17b6587794971e   

                                                text  dims  
0  3:03 "You can clap about that all you want. En...  1024  
1  I respect everyone's psition on the matter, in...  1024  
2  That last world Sam describes... If those kids...  1024  
3  Sam Altman never really answer. Maybe I am not...  1024  
4  AI is not about ego and who gets credit for wh...  1024  


In [23]:
def carregar_embeddings_do_json(nome_arquivo):
    """
    Carrega embeddings de um arquivo JSON no formato especificado.

    Args:
        nome_arquivo (str): O caminho para o arquivo JSON.

    Returns:
        list or None: Uma lista de listas de floats (os embeddings),
                     ou None se ocorrer um erro ao carregar o arquivo.
    """
    try:
        with open(nome_arquivo, 'r') as f:
            data = json.load(f)
            if "embedding" in data and isinstance(data["embedding"], list):
                return data["embedding"]
            else:
                print(f"Formato inválido no arquivo '{nome_arquivo}'. Esperava uma lista na chave 'embedding'.")
                return None
    except FileNotFoundError:
        print(f"Arquivo '{nome_arquivo}' não encontrado.")
        return None
    except json.JSONDecodeError:
        print(f"Erro ao decodificar JSON do arquivo '{nome_arquivo}'.")
        return None
    except Exception as e:
        print(f"Ocorreu um erro ao carregar o arquivo '{nome_arquivo}': {e}")
        return None

In [27]:
# Supondo que você tem um embedding de referência
reference_embedding = carregar_embeddings_do_json(nome_arquivo=r'C:\Users\fuedj\Documents\Code\Embeddings_Cluster\embedding.json')  # Substitua por um embedding real
len(reference_embedding)

1024

In [48]:
def cosine_similarity(a: List[float], b: List[float]) -> float:
    """Calcula a similaridade de cossenos entre dois vetores."""
    a_np = np.array(a)
    b_np = np.array(b)
    dot_product = np.dot(a_np, b_np)
    norm_a = np.linalg.norm(a_np)
    norm_b = np.linalg.norm(b_np)
    if norm_a == 0 or norm_b == 0:
        return 0.0
    return dot_product / (norm_a * norm_b)

In [52]:
def encontrar_embeddings_similares_python(
    reference_embedding: List[float], database_file: str = EMBEDDINGS_FILE, top_n: int = 3
) -> List[tuple]:
    """
    Encontra os top N embeddings mais similares no banco de dados usando a função Python.

    Args:
        reference_embedding: O embedding de referência (lista de floats).
        database_file: O caminho para o arquivo do banco de dados DuckDB.
        top_n: O número de embeddings mais similares a serem retornados.

    Returns:
        Uma lista de tuplas, onde cada tupla contém (id, text, similarity).
    """
    resultados_similaridade = []
    try:
        with duckdb.connect(database_file) as conn:
            # Recupera todos os ids, textos e embeddings do banco de dados
            data = conn.execute("SELECT id, text, embedding FROM embeddings").fetchall()

            for id, text, embedding in data:
                if embedding and len(embedding) > 0:  # Verifica se a lista não é vazia
                    similarity = cosine_similarity(reference_embedding, embedding)
                    resultados_similaridade.append((id, text, similarity))
                else:
                    print(f"Aviso: Encontrou embedding vazio para ID {id}.")

            # Ordena os resultados por similaridade em ordem decrescente
            resultados_ordenados = sorted(resultados_similaridade, key=lambda item: item[2], reverse=True)

            return resultados_ordenados[:top_n]

    except duckdb.Error as e:
        print(f"Erro ao acessar o banco de dados: {e}")
        return []

In [54]:
# Encontra os embeddings mais similares usando a função Python
similares = encontrar_embeddings_similares_python(reference_embedding)

if similares:
    print("\nComentários mais similares (calculados em Python):")
    for id, text, similarity in similares:
        print(f"ID: {id[:5]}, Texto: {text}..., Similaridade: {similarity:.4f}")
else:
    print("Não foram encontrados embeddings similares.")

Aviso: Encontrou embedding vazio para ID 68b329da9893e34099c7d8ad5cb9c940.

Comentários mais similares (calculados em Python):
ID: a25a4, Texto: Not thrilled with Altman's defensiveness. He needs to spend more time in a humility chamber of sorts where he has no control over future but others know his fate in advance. He's asking all of us to trust the future during a time when our government is rolling backward through centuries of human suffering. Asking questions allows agency over our lives.
..., Similaridade: 0.8666
ID: 1dbe7, Texto: Good stuff. Altman should be challenged, and it's a cool brag to be challenged. He's doing something with massive impact
..., Similaridade: 0.8392
ID: 992d6, Texto: In this interview it shone through quite clearly that Sam Altmans ego and petulence are causes for concern. He couldnt handle the fair and reasonable questions proposed to him. He resorted to immature gestures and comments. This is likely what those who exited OpenAI saw daily.We should be 

In [55]:
import duckdb

EMBEDDINGS_FILE = "data/embeddings.duckdb"  # Substitua pelo seu nome de arquivo

id_to_access = "68b329da9893e34099c7d8ad5cb9c940"

try:
    with duckdb.connect(EMBEDDINGS_FILE) as conn:
        result = conn.execute("SELECT * FROM embeddings WHERE id = ?", [id_to_access]).fetchdf()
        if not result.empty:
            print(result.to_markdown(index=False))  # Imprime o resultado como uma tabela Markdown
        else:
            print(f"Nenhum dado encontrado para o ID '{id_to_access}'.")
except duckdb.Error as e:
    print(f"Erro ao acessar o banco de dados: {e}")

| id                               | embedding   | text   | created_at                 |
|:---------------------------------|:------------|:-------|:---------------------------|
| 68b329da9893e34099c7d8ad5cb9c940 | []          |        | 2025-05-01 11:29:07.615000 |


In [ ]:
def cosine_similarity(a, b):
    dot_product = sum(x * y for x, y in zip(a, b))
    magnitude_a = sum(x * x for x in a) ** 0.5
    magnitude_b = sum(x * x for x in b) ** 0.5
    return dot_product / (magnitude_a * magnitude_b)
    

similar = conn.execute("""
SELECT id, text,
       array_cosine_similarity(embedding, ?) as similarity
FROM embeddings
WHERE CARDINALITY(embedding) = ?
ORDER BY similarity DESC
LIMIT 3
""", [reference_embedding, len(reference_embedding)]).fetchdf()

print("\nComentários mais similares:")
print(similar)

ParserException: Parser Error: syntax error at or near "FLOAT"

In [29]:
stats = conn.execute("""
SELECT 
    MIN(array_length(embedding)) as min_dims,
    MAX(array_length(embedding)) as max_dims,
    AVG(array_length(embedding)) as avg_dims
FROM embeddings
""").fetchdf()
print("\nEstatísticas de dimensões:")
print(stats)


Estatísticas de dimensões:
   min_dims  max_dims     avg_dims
0         0      1024  1023.153019


In [30]:
import time

start = time.time()
result = conn.execute("SELECT COUNT(*) FROM embeddings").fetchone()
print(f"Consulta COUNT executada em {time.time() - start:.4f} segundos")

Consulta COUNT executada em 0.0000 segundos


In [31]:
# Encontre embeddings com alta dimensionalidade
high_dim = conn.execute("""
SELECT id, array_length(embedding) as dims
FROM embeddings
WHERE array_length(embedding) > ?
ORDER BY dims DESC
LIMIT 5
""", [768]).fetchdf()  # Assumindo dimensão esperada de 768
print("\nEmbeddings de alta dimensão:")
print(high_dim)


Embeddings de alta dimensão:
                                 id  dims
0  cb0f2e74ebcbfb0a21f320fa6ef0b1eb  1024
1  a1b8f2efb813c2025ce7289e412e1ad2  1024
2  1e642a39f0b419accc96ed1fc67e416b  1024
3  a7e967f87c4765a321527844d01727c2  1024
4  b3967ad0b125798f6a17b6587794971e  1024


In [32]:
problematic = conn.execute("""
SELECT id, text 
FROM embeddings 
WHERE text LIKE '%#%'
LIMIT 5
""").fetchdf()
print("\nComentários contendo '#':")
print(problematic)


Comentários contendo '#':
                                 id  \
0  2e32f8b6e714f151093252e14b4e5347   
1  fdc0744f0ce6a24e18ff1a1f33c40687   

                                                text  
0  I think we should use WiFi. To act as radar. N...  
1  At 15;37 Sam tells a blatant lie. Here is what...  
